# Step 2 — Data Loading & Preprocessing

Adult Census dataset. Outputs:
- `data/adult_clean.csv` — full cleaned dataset (no split)
- `data/adult_train.csv` — 80% split, used to train synthesisers
- `data/adult_test.csv` — 20% holdout, non-members for TAPAS MIA

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split

## Load

In [26]:
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

df = pd.read_csv('data/adult.data', names=columns, skipinitialspace=True)
print(f'Loaded: {df.shape}')
df.head()

Loaded: (32561, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## Clean

In [27]:
# Strip whitespace from all string columns
str_cols = df.select_dtypes('object').columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

# Drop rows with missing values (encoded as '?')
n_before = len(df)
df = df.replace('?', pd.NA).dropna()
print(f'Dropped {n_before - len(df)} rows with missing values ({(n_before - len(df)) / n_before:.1%}). Remaining: {len(df)}')

Dropped 2399 rows with missing values (7.4%). Remaining: 30162


In [28]:
# Drop fnlwgt (census sampling weight, not a real feature)
# Drop education (redundant with education_num)
df = df.drop(columns=['fnlwgt', 'education'])

# Deduplicate after column drops — rows that only differed in fnlwgt become identical once it's removed
n_before = len(df)
df = df.drop_duplicates()
print(f'Dropped {n_before - len(df)} duplicate rows after column drops. Remaining: {len(df)}')
print(f'Columns: {df.columns.tolist()}')

Dropped 3258 duplicate rows after column drops. Remaining: 26904
Columns: ['age', 'workclass', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']


## Column schema

Explicit typing needed for Synthcity's DataLoader and for TAPAS attack config.

In [29]:
CONTINUOUS_COLS = ['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
CATEGORICAL_COLS = ['workclass', 'marital_status', 'occupation', 'relationship',
                    'race', 'sex', 'native_country', 'income']
TARGET_COL = 'income'  # used for utility eval (downstream classifier) and TAPAS attribute inference

# Enforce dtypes
df[CONTINUOUS_COLS] = df[CONTINUOUS_COLS].astype(float)
df[CATEGORICAL_COLS] = df[CATEGORICAL_COLS].astype('category')

print(df.dtypes)
print(f'\nTarget classes: {df[TARGET_COL].unique().tolist()}')

age                float64
workclass         category
education_num      float64
marital_status    category
occupation        category
relationship      category
race              category
sex               category
capital_gain       float64
capital_loss       float64
hours_per_week     float64
native_country    category
income            category
dtype: object

Target classes: ['<=50K', '>50K']


## Train / test split

80/20 split. Synthesisers only ever see `adult_train`. The holdout (`adult_test`) provides genuine non-members for TAPAS membership inference.

In [30]:
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df[TARGET_COL])
print(f'Train: {train.shape}, Test: {test.shape}')
print(f'\nTarget distribution (train):\n{train[TARGET_COL].value_counts(normalize=True).round(3)}')
print(f'\nTarget distribution (test):\n{test[TARGET_COL].value_counts(normalize=True).round(3)}')

Train: (21523, 13), Test: (5381, 13)

Target distribution (train):
income
<=50K    0.744
>50K     0.256
Name: proportion, dtype: float64

Target distribution (test):
income
<=50K    0.744
>50K     0.256
Name: proportion, dtype: float64


## Save

In [31]:
df.to_csv('data/adult_clean.csv', index=False)
train.to_csv('data/adult_train.csv', index=False)
test.to_csv('data/adult_test.csv', index=False)
print('Saved: data/adult_clean.csv, data/adult_train.csv, data/adult_test.csv')

Saved: data/adult_clean.csv, data/adult_train.csv, data/adult_test.csv


## Quick sanity check

In [32]:
train_check = pd.read_csv('data/adult_train.csv')
test_check = pd.read_csv('data/adult_test.csv')

# Merge on all columns — any matches mean a row appears in both splits
overlap = pd.merge(train_check, test_check, how='inner')
print(f'Rows shared between train and test: {len(overlap)} (expected 0)')
print(f'Train shape: {train_check.shape}')
print(f'Test shape:  {test_check.shape}')
train_check.describe()

Rows shared between train and test: 0 (expected 0)
Train shape: (21523, 13)
Test shape:  (5381, 13)


,age,education_num,capital_gain,capital_loss,hours_per_week
count,21523.000000,21523.000000,21523.000000,21523.000000,21523.000000
mean,39.010919,10.145937,1238.288993,96.877155,41.183478
std,13.171149,2.626226,8010.669410,422.333853,12.350657
min,17.000000,1.000000,0.000000,0.000000,1.000000
25%,29.000000,9.000000,0.000000,0.000000,40.000000
50%,38.000000,10.000000,0.000000,0.000000,40.000000
75%,48.000000,13.000000,0.000000,0.000000,45.000000
max,90.000000,16.000000,99999.000000,4356.000000,99.000000
